# 🚀 HUẤN LUYỆN VÀ ĐÁNH GIÁ 3 MÔ HÌNH TRANSFORMER TRÊN GOOGLE COLAB (GPU)

**Đề tài:** Phát hiện phát ngôn độc hại trên mạng xã hội tiếng Việt

**Mô hình:** PhoBERT-base-v2, ViDeBERTa-base, XLM-RoBERTa-base
**Tác vụ:** Tác vụ 1 (TOXIC/NON-TOXIC) & Tác vụ 2 (EXPLICIT/IMPLICIT)

### 📌 Bước 1: Kiểm tra GPU & Cài đặt thư viện

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('GPU Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️ CẢNH BÁO: Chưa bật GPU! Vào Runtime -> Change runtime type -> Chọn T4 GPU.')

In [ ]:
!pip install -q transformers pyvi accelerate scikit-learn pandas numpy matplotlib seaborn tqdm

### 📌 Bước 2: Tải hoặc Giải nén dữ liệu (`Model_final_v2.zip` hoặc `data_splits/`)

In [ ]:
import os
from pathlib import Path

if os.path.exists('Model_final_v2.zip'):
    !unzip -q -o Model_final_v2.zip
    print('Đã giải nén Model_final_v2.zip thành công!')

data_dir = Path('Model_final_v2/data_splits') if os.path.exists('Model_final_v2/data_splits') else Path('data_splits')
for f in ['train.csv', 'dev.csv', 'test.csv']:
    p = data_dir / f
    if p.exists():
        print(f'✅ Tìm thấy: {p} ({p.stat().st_size / 1e6:.2f} MB)')
    else:
        print(f'❌ Chưa tìm thấy {f}! Vui lòng kiểm tra thư mục {data_dir}')

### 📌 Bước 3: Định nghĩa hàm huấn luyện tối ưu GPU

In [ ]:
import os, sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import pyvi.ViTokenizer as vi_tokenizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128, use_pyvi=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.use_pyvi = use_pyvi

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        if self.use_pyvi:
            text = vi_tokenizer.tokenize(text)
        encoding = self.tokenizer(text, truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def compute_metrics(y_true, y_pred):
    return {
        'Accuracy': round(float(accuracy_score(y_true, y_pred)), 4),
        'Macro_Precision': round(float(precision_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'Macro_Recall': round(float(recall_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'Macro_F1': round(float(f1_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'Weighted_F1': round(float(f1_score(y_true, y_pred, average='weighted', zero_division=0)), 4)
    }

def run_model_training(model_name, ckpt, use_pyvi, task_name='task1', epochs=5, batch_size=32, lr=2e-5):
    set_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data_dir = Path('Model_final_v2/data_splits') if os.path.exists('Model_final_v2/data_splits') else Path('data_splits')

    train_df = pd.read_csv(data_dir / 'train.csv', dtype='string', keep_default_na=False, encoding='utf-8-sig')
    dev_df = pd.read_csv(data_dir / 'dev.csv', dtype='string', keep_default_na=False, encoding='utf-8-sig')
    test_df = pd.read_csv(data_dir / 'test.csv', dtype='string', keep_default_na=False, encoding='utf-8-sig')

    if task_name == 'task1':
        label_col = 'toxicity'
        label_map = {'NON-TOXIC': 0, 'TOXIC': 1}
        target_names = ['NON-TOXIC', 'TOXIC']
        w0 = len(train_df) / (2.0 * (train_df['toxicity'] == 'NON-TOXIC').sum())
        w1 = len(train_df) / (2.0 * (train_df['toxicity'] == 'TOXIC').sum())
    else:
        train_df = train_df[train_df['toxicity'] == 'TOXIC'].reset_index(drop=True)
        dev_df = dev_df[dev_df['toxicity'] == 'TOXIC'].reset_index(drop=True)
        test_df = test_df[test_df['toxicity'] == 'TOXIC'].reset_index(drop=True)
        label_col = 'expression_type'
        label_map = {'EXPLICIT': 0, 'IMPLICIT': 1}
        target_names = ['EXPLICIT', 'IMPLICIT']
        w0 = len(train_df) / (2.0 * (train_df['expression_type'] == 'EXPLICIT').sum())
        w1 = len(train_df) / (2.0 * (train_df['expression_type'] == 'IMPLICIT').sum())

    class_weights = [w0, w1]
    print(f'\n>>> [TRAINING {model_name.upper()}] - {task_name.upper()} ({len(train_df)} train, {len(dev_df)} dev, {len(test_df)} test) <<<')

    tokenizer = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2).to(device)

    train_loader = DataLoader(TextDataset(train_df['comment'].tolist(), [label_map[l] for l in train_df[label_col]], tokenizer, 128, use_pyvi), batch_size=batch_size, shuffle=True, pin_memory=True)
    dev_loader = DataLoader(TextDataset(dev_df['comment'].tolist(), [label_map[l] for l in dev_df[label_col]], tokenizer, 128, use_pyvi), batch_size=batch_size*2, shuffle=False)
    test_loader = DataLoader(TextDataset(test_df['comment'].tolist(), [label_map[l] for l in test_df[label_col]], tokenizer, 128, use_pyvi), batch_size=batch_size*2, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float).to(device))
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_dev_macro_f1 = -1.0
    best_model_state = None
    best_epoch = -1

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_loss += loss.item() * len(labels)
        train_loss /= len(train_df)

        # Dev Evaluation
        model.eval()
        dev_preds, dev_targets = [], []
        with torch.no_grad():
            for batch in dev_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                dev_preds.extend(preds)
                dev_targets.extend(labels.cpu().numpy())

        dev_m = compute_metrics(dev_targets, dev_preds)
        print(f'Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Dev Acc: {dev_m["Accuracy"]:.4f} | Dev Macro-F1: {dev_m["Macro_F1"]:.4f}')

        if dev_m['Macro_F1'] > best_dev_macro_f1:
            best_dev_macro_f1 = dev_m['Macro_F1']
            best_epoch = epoch
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}

    # Test Set Evaluation on Best Checkpoint
    print(f'Evaluating Best Checkpoint (Epoch {best_epoch}, Dev Macro-F1: {best_dev_macro_f1:.4f}) on LOCKED TEST SET...')
    model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    model.eval()
    test_preds, test_probs, test_targets = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            test_probs.extend(probs)
            test_preds.extend(preds)
            test_targets.extend(labels.cpu().numpy())

    test_m = compute_metrics(test_targets, test_preds)
    cm = confusion_matrix(test_targets, test_preds)
    cm_df = pd.DataFrame(cm, index=[f'Actual_{n}' for n in target_names], columns=[f'Pred_{n}' for n in target_names])
    print(f'\nFINAL TEST RESULTS: {model_name} ({task_name}):\n', json.dumps(test_m, indent=2))
    print('\nConfusion Matrix:\n', cm_df)
    return test_m, cm_df

### 📌 Bước 4: Chạy Fine-Tuning PhoBERT-base-v2

In [ ]:
phobert_t1, cm_phobert_t1 = run_model_training('PhoBERT', 'vinai/phobert-base-v2', use_pyvi=True, task_name='task1', epochs=5, batch_size=32)
phobert_t2, cm_phobert_t2 = run_model_training('PhoBERT', 'vinai/phobert-base-v2', use_pyvi=True, task_name='task2', epochs=5, batch_size=32)

### 📌 Bước 5: Chạy Fine-Tuning ViDeBERTa-base

In [ ]:
videberta_t1, cm_videberta_t1 = run_model_training('ViDeBERTa', 'Fsoft-AIC/videberta-base', use_pyvi=False, task_name='task1', epochs=5, batch_size=32)
videberta_t2, cm_videberta_t2 = run_model_training('ViDeBERTa', 'Fsoft-AIC/videberta-base', use_pyvi=False, task_name='task2', epochs=5, batch_size=32)

### 📌 Bước 6: Chạy Fine-Tuning XLM-RoBERTa-base

In [ ]:
xlmr_t1, cm_xlmr_t1 = run_model_training('XLM-R', 'FacebookAI/xlm-roberta-base', use_pyvi=False, task_name='task1', epochs=5, batch_size=32)
xlmr_t2, cm_xlmr_t2 = run_model_training('XLM-R', 'FacebookAI/xlm-roberta-base', use_pyvi=False, task_name='task2', epochs=5, batch_size=32)

### 📌 Bước 7: Tự động Đọc TextCNN từ file và Merge Benchmark 4 Mô hình (KHÔNG HARDCODE)

In [ ]:
# Đọc metrics TextCNN từ file kết quả thực tế
res_t1_path = Path('Model_final_v2/results/test_results_task1.csv') if os.path.exists('Model_final_v2/results/test_results_task1.csv') else Path('results/test_results_task1.csv')
res_t2_path = Path('Model_final_v2/results/test_results_task2.csv') if os.path.exists('Model_final_v2/results/test_results_task2.csv') else Path('results/test_results_task2.csv')

t1_base = pd.read_csv(res_t1_path)
t2_base = pd.read_csv(res_t2_path)
textcnn_t1_row = t1_base[t1_base['Model'].str.contains('TextCNN')].iloc[0].to_dict()
textcnn_t2_row = t2_base[t2_base['Model'].str.contains('TextCNN')].iloc[0].to_dict()

task1_summary = pd.DataFrame([
    textcnn_t1_row,
    {'Model': 'PhoBERT-base-v2', 'Architecture': 'Pretrained RoBERTa (Monolingual)', 'Status': 'COMPLETED', **phobert_t1},
    {'Model': 'ViDeBERTa-base', 'Architecture': 'Pretrained DeBERTa (Disentangled)', 'Status': 'COMPLETED', **videberta_t1},
    {'Model': 'XLM-RoBERTa-base', 'Architecture': 'Pretrained RoBERTa (Multilingual)', 'Status': 'COMPLETED', **xlmr_t1}
])
print('=== BẢNG SO SÁNH TÁC VỤ 1 (TOXIC / NON-TOXIC) ===')
display(task1_summary)
task1_summary.to_csv('colab_test_results_task1.csv', index=False, encoding='utf-8-sig')

task2_summary = pd.DataFrame([
    textcnn_t2_row,
    {'Model': 'PhoBERT-base-v2', 'Architecture': 'Pretrained RoBERTa (Monolingual)', 'Status': 'COMPLETED', **phobert_t2},
    {'Model': 'ViDeBERTa-base', 'Architecture': 'Pretrained DeBERTa (Disentangled)', 'Status': 'COMPLETED', **videberta_t2},
    {'Model': 'XLM-RoBERTa-base', 'Architecture': 'Pretrained RoBERTa (Multilingual)', 'Status': 'COMPLETED', **xlmr_t2}
])
print('=== BẢNG SO SÁNH TÁC VỤ 2 (EXPLICIT / IMPLICIT) ===')
display(task2_summary)
task2_summary.to_csv('colab_test_results_task2.csv', index=False, encoding='utf-8-sig')

### 📌 Bước 8: Tải xuống Toàn bộ Kết quả Thực nghiệm về máy

In [ ]:
from google.colab import files
import zipfile

with zipfile.ZipFile('colab_final_results.zip', 'w') as z:
    for f in ['colab_test_results_task1.csv', 'colab_test_results_task2.csv']:
        if os.path.exists(f):
            z.write(f)
print('Đang kích hoạt tải file colab_final_results.zip...')
files.download('colab_final_results.zip')